In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)


def parse_survival_info(s: str):
    # Expect format:
    # "survival: recurrence:1, progression-free-survival_days:144, live, overall-survival_days:234"
    # Positions are fixed: [PFS_event, PFS_time, OS_event, OS_time]
    after = s.split('survival:', 1)[-1].strip()
    parts = [p.strip() for p in after.split(',')]
    # ['recurrence:1', 'progression-free-survival_days:144', 'live', 'overall-survival_days:234']

    pfs_event = parts[0].split(':', 1)[1].strip()       # '1'
    pfs_time = parts[1].split(':', 1)[1].strip()        # '144'
    os_event = parts[2]                                  # 'live'
    os_time = parts[3].split(':', 1)[1].strip()         # '234'

    return {
        'progression-free-survival_days': int(pfs_time),
        'recurrence': int(pfs_event),
        'overall-survival_days': int(os_time),
        'survival': os_event,
    }



# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {"sample_title": sample_title}
    for item in value.metadata['characteristics_ch1']:
        
        # specific logic for this datasets
        if 'survival' in item:
            for k, v in parse_survival_info(item).items():
                metadata_object[k] = v
        else:   
            k, v = item.split(':')
            metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

# Use NCBI provided expression data


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined = metadata_df.join(df_raw_counts, how='inner')


df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

print(df_combined.shape)
df_combined.head()

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_93791/4229065046.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


(10, 39384)


,sample_title,tissue,tumor type,Sex,progression-free-survival_days,recurrence,overall-survival_days,survival,DDX11L1,WASH7P,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM4063814,D01,lung,adenocarcinoma,female,144,1,234,live,0,0,...,3,0,0,0,3,0,0,0,0,0
GSM4063815,D03,lung,adenocarcinoma,male,1088,0,1088,live,0,0,...,32,0,0,0,20,3,1,28,0,0
GSM4063816,D06,lung,Large cell neuroendocarine carcinoma,male,253,1,262,dead,0,0,...,0,0,0,0,0,0,0,0,0,0
GSM4063817,D07,lung,adenocarcinoma,female,259,1,259,dead,0,0,...,0,0,0,0,0,0,0,0,0,0
GSM4063818,D08,lung,Large cell neuroendocarine carcinoma,male,191,1,376,dead,0,0,...,33,0,0,0,25,3,1,24,1,0


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated


# Use author provided expression data


In [4]:
metadata_df = metadata_df.reset_index()
metadata_df = metadata_df.set_index('sample_title')

tmp_url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE136961&format=file&file=GSE136961%5FTPM%2Etsv%2Egz"
df_tpm: pd.DataFrame = pd.read_csv(tmp_url, sep='\t', index_col=0, compression='gzip')
df_tpm = df_tpm.T
df_tpm.index.name = "samples"
df_tpm.columns.name = None

raw_counts_url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE136961&format=file&file=GSE136961%5Fraw%5Fcount%2Etsv%2Egz"
df_raw_count: pd.DataFrame = pd.read_csv(raw_counts_url, sep='\t', index_col=0, compression='gzip')
df_raw_count = df_raw_count.set_index("Target")
df_raw_count = df_raw_count.drop(columns=['NCBI_NAME', 'ENTREZ_GENE_ID', 'NCBI_ACCESSION', 'GENE_FUNCTION'])
df_raw_count= df_raw_count.T
df_raw_count.index.name = "samples"
df_raw_count.columns.name = None

df_tpm = metadata_df.join(df_tpm, how='inner')
df_tpm.reset_index(inplace=True)
# INFO: Uncomment to save the data to disk
# df_tpm.to_csv(f'{GEO_ID}_TPM.csv', index=False)

df_raw_count = metadata_df.join(df_raw_count, how='inner')
df_raw_count.reset_index(inplace=True)
# INFO: Uncomment to save the data to disk
# df_raw_count.to_csv(f'{GEO_ID}_RAW_COUNTS.csv', index=False)

df_tpm

,index,samples,tissue,tumor type,Sex,progression-free-survival_days,recurrence,overall-survival_days,survival,ABCF1_11741283,...,TUBB_369479,TWIST1_9291033,TYROBP_350455,VCAM1_829933,VEGFA_17121817,VTCN1_49157,XAGE1B_469547,ZAP70_673781,ZBTB46_13361440,ZEB1_74175
0,D01,GSM4063814,lung,adenocarcinoma,female,144,1,234,live,7904.660731,...,7156.847758,295.319612,3941.184729,1629.126568,4213.892063,0.029250,2895.428480,112.917347,11.004809,2.796664
1,D03,GSM4063815,lung,adenocarcinoma,male,1088,0,1088,live,12693.990956,...,17494.545463,104.867641,7547.050168,437.455913,5533.307003,0.000000,8923.206144,17.352750,12.143592,0.541179
2,D06,GSM4063816,lung,Large cell neuroendocarine carcinoma,male,253,1,262,dead,5698.365555,...,8015.402876,218.309595,6806.169733,560.256816,983.944743,0.000000,0.000000,63.157956,5.689701,3.734474
3,D07,GSM4063817,lung,adenocarcinoma,female,259,1,259,dead,9359.068430,...,10768.720850,36.841878,5782.153455,718.671845,3704.324473,1.362722,5410.545076,37.687740,16.640644,3.013758
4,D08,GSM4063818,lung,Large cell neuroendocarine carcinoma,male,191,1,376,dead,17693.809230,...,30609.787679,54.599033,5583.086780,1167.239762,5399.151756,0.000000,1105.073930,15.804681,26.520098,10.275894
5,D09,GSM4063819,lung,adenocarcinoma,male,362,1,362,dead,10229.757126,...,7125.718698,602.037374,3114.115585,1434.474367,2193.036880,0.011050,72.876390,53.033393,0.373042,6.822228
6,N05,GSM4063820,lung,adenocarcinoma,male,64,1,260,dead,10386.178373,...,5984.397485,6.705798,3820.412505,858.250049,9305.948635,0.018791,479.890871,36.713504,9.348325,3.855311
7,N06,GSM4063821,lung,Large cell neuroendocarine carcinoma,male,54,1,147,dead,11339.813449,...,24876.017441,325.196840,2955.345361,761.499130,6370.790481,0.000000,3.290032,16.398663,4.105875,5.706888
8,N08,GSM4063822,lung,squamous cell carcinoma,male,28,1,329,dead,3540.964719,...,10031.480821,152.762380,221.044316,96.060796,5547.233739,0.698795,0.000000,2.089855,0.762390,1.822591
9,N10,GSM4063823,lung,adenocarcinoma-squamous cell carcinoma,male,70,1,70,dead,5957.135903,...,4801.114117,20.575268,603.318130,58.691501,902.180440,4.287435,0.000000,1.127161,1.462102,0.232107
